In [1]:
import os, time, shutil, glob
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException, NoAlertPresentException, WebDriverException
from datetime import datetime, timedelta
from selenium.webdriver.common.keys import Keys

# ── CONFIG ─────────────────────────────────────────────────────
CHROMEDRIVER   = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE\chromedriver-win64\chromedriver.exe"
SOURCE_FOLDER  = r"C:\temp\expedia_downloads"
IEX_DEST_DIR   = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE\current_iex"
os.makedirs(IEX_DEST_DIR, exist_ok=True)

NICE_URL_GENERATE = "https://cnxnice02b.nicecloudsvc.com/wfm/supervisor/reports-generate"
NICE_URL_VIEW     = "https://cnxnice02b.nicecloudsvc.com/wfm/supervisor/reports-view"
NICE_REPORT_URL   = "https://cnxnice02b.nicecloudsvc.com/supv/reportAction.mvc?schRptOid=8aa89bca8b4b614b018d4b88862d476c"
NICE_USER         = "huuchinh.nguyen@concentrix.com"
NICE_PASS         = "Vuthihongtham@130499"

# ── NICE date range (Monday → Sunday this week) ───────────────
_today    = datetime.now()
_monday   = _today - timedelta(days=_today.weekday())
_sunday   = _monday + timedelta(days=6)
NICE_FROM = _monday.strftime("%#m/%#d/%y")
NICE_TO   = _sunday.strftime("%#m/%#d/%y")
NICE_FILE = _monday.strftime("%Y_%m_%d") + ".xlsx"

# ── HELPERS ────────────────────────────────────────────────────
def nice_login(driver, wait):
    try:
        # Step 0: Click "Concentrix Authentication"
        try:
            cnx_auth = WebDriverWait(driver, 8).until(
                EC.element_to_be_clickable((
                    By.XPATH,
                    '//span[contains(@class,"largeTextNoWrap") and '
                    'contains(text(),"Concentrix Authentication")]'
                ))
            )
            driver.execute_script("arguments[0].click();", cnx_auth)
            print("  ✅ Clicked 'Concentrix Authentication'")
            time.sleep(3)
        except TimeoutException:
            pass

        # Detect login page: USERNAME tab button
        username_tab = WebDriverWait(driver, 8).until(
            EC.element_to_be_clickable((
                By.XPATH,
                '//button[@aria-label="Passwordless users, login here." '
                'and normalize-space(text())="Username"]'
            ))
        )
        print("  🔑 NICE login page detected")

        # Click USERNAME tab
        driver.execute_script("arguments[0].click();", username_tab)
        time.sleep(1)

        # Remember me checkbox
        try:
            cb = driver.find_element(By.ID, "checkboxRememberMe")
            if not cb.is_selected():
                driver.execute_script("arguments[0].click();", cb)
                print("  ✅ Checked 'Remember me on this device'")
            time.sleep(0.5)
        except Exception:
            pass

        # Enter username
        user_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "username"))
        )
        driver.execute_script("arguments[0].value = '';", user_input)
        user_input.click()
        user_input.send_keys(Keys.CONTROL + "a")
        user_input.send_keys(Keys.DELETE)
        time.sleep(0.3)
        user_input.send_keys(NICE_USER)
        print(f"  ✅ Entered username: {NICE_USER}")
        time.sleep(0.5)

        # Click Next (username submit)
        next_btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[type="submit"]'))
        )
        driver.execute_script("arguments[0].click();", next_btn)
        print("  ✅ Clicked Next (username)")
        time.sleep(2)

        # Click "Password / Login with Password"
        try:
            pwd_card = WebDriverWait(driver, 8).until(
                EC.element_to_be_clickable((
                    By.XPATH,
                    '//*[contains(@class,"jss142") and normalize-space(text())="Password"]'
                    '/ancestor::div[contains(@class,"jss140")]'
                ))
            )
            driver.execute_script("arguments[0].click();", pwd_card)
            print("  ✅ Clicked 'Password' method card")
            time.sleep(2)
        except TimeoutException:
            pass

        # Wait for password field
        pwd_input = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.ID, "password"))
        )
        driver.execute_script("arguments[0].value = '';", pwd_input)
        pwd_input.click()
        pwd_input.send_keys(Keys.CONTROL + "a")
        pwd_input.send_keys(Keys.DELETE)
        time.sleep(0.3)
        pwd_input.send_keys(NICE_PASS)
        print("  ✅ Entered password")
        time.sleep(0.5)

        # Click Next (password submit)
        next_btn2 = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[type="submit"]'))
        )
        driver.execute_script("arguments[0].click();", next_btn2)
        print("  ✅ Clicked Next (password)")

        # Wait for redirect to NICE WFM
        WebDriverWait(driver, 30).until(
            lambda d: "nicecloudsvc.com/wfm" in d.current_url
        )
        print("  🎉 NICE login successful")

    except TimeoutException:
        print("  ✅ NICE already authenticated — no login needed")

def nice_check_timeout(driver, fallback_url, max_retry=3):
    for attempt in range(max_retry):
        if "timeout" in driver.current_url.lower():
            print(f"  ⚠️ Session timed out — reloading (attempt {attempt+1})...")
            driver.get(fallback_url)
            time.sleep(8)
        else:
            return True
    print("  ❌ Still timed out after retries")
    return False

# ── INIT DRIVER ────────────────────────────────────────────────
chrome_options = Options()
chrome_options.add_argument(r"--user-data-dir=C:/temp/nice_chrome_profile")
chrome_options.add_argument(r"--profile-directory=Default")
chrome_options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(CHROMEDRIVER), options=chrome_options)
wait   = WebDriverWait(driver, 15)

print(f"\n{'═'*55}")
print(f"🚀 NICE IEX Bot started: {datetime.now().strftime('%d-%b-%Y %H:%M:%S')}")
print(f"📅 Week: {NICE_FROM} → {NICE_TO} | File: {NICE_FILE}")
print(f"{'═'*55}")

try:
    # ── 1. Navigate + Login ───────────────────────────────────
    print("\n[1/4] NICE WFM Login...")
    driver.get(NICE_URL_GENERATE)
    time.sleep(8)
    nice_login(driver, wait)

    if not nice_check_timeout(driver, NICE_URL_GENERATE):
        raise RuntimeError("NICE session timeout — cannot proceed")

    # ── 2. Verify page loaded ─────────────────────────────────
    print("\n[2/4] Verifying NICE page...")
    retry, found = 0, False
    while retry < 5:
        try:
            gen_link = WebDriverWait(driver, 10).until(EC.presence_of_element_located(
                (By.XPATH, '//a[@title="Generate" and contains(@class,"sub-menu-item")]')
            ))
            if gen_link.is_displayed():
                print(f"  ✅ NICE page ready (attempt {retry+1})")
                found = True; break
            raise Exception("not visible")
        except Exception:
            retry += 1
            print(f"  ⏳ Attempt {retry}/5...")
            time.sleep(5)
            driver.execute_script(f"window.open('{NICE_URL_GENERATE}','_blank');")
            driver.switch_to.window(driver.window_handles[-1])
            nice_login(driver, wait)

    if not found:
        raise RuntimeError("NICE page did not load after 5 attempts")

    # ── 3. Set date range + Generate ─────────────────────────
    print(f"\n[3/4] Generating report {NICE_FROM} → {NICE_TO}...")
    driver.get(NICE_REPORT_URL); time.sleep(5)

    if not nice_check_timeout(driver, NICE_URL_GENERATE):
        raise RuntimeError("Session timeout on report page")

    inp_s = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "stAbsDate")))
    inp_s.clear(); inp_s.send_keys(NICE_FROM); time.sleep(1)

    inp_e = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "endAbsDate")))
    inp_e.clear(); inp_e.send_keys(NICE_TO); time.sleep(1)

    WebDriverWait(driver, 10).until(EC.element_to_be_clickable(
        (By.XPATH, "//input[@type='submit' and @value='Generate']")
    )).click()
    print("  ✅ Generate clicked")
    time.sleep(3)

    try: driver.switch_to.alert.accept()
    except NoAlertPresentException: pass

    print("  ⏳ Waiting 60s for generation...")
    time.sleep(60)

    # ── 4. Download + Move ────────────────────────────────────
    print("\n[4/4] Downloading Agent Schedules...")
    driver.get(NICE_URL_VIEW)

    if not nice_check_timeout(driver, NICE_URL_GENERATE):
        raise RuntimeError("Session timeout on view page")

    wait_n = WebDriverWait(driver, 20)
    iframe = wait_n.until(EC.presence_of_element_located((By.CLASS_NAME, "legacy-wrapper")))
    driver.switch_to.frame(iframe)

    wait_n.until(EC.element_to_be_clickable(
        (By.XPATH, '//input[@type="submit" and @value="Refresh"]')
    )).click()
    time.sleep(10)

    wait_n.until(EC.element_to_be_clickable((By.LINK_TEXT, "Agent Schedules"))).click()
    print("  ✅ Clicked 'Agent Schedules'")
    driver.switch_to.default_content()
    time.sleep(15)

    dst_path = os.path.join(IEX_DEST_DIR, NICE_FILE)
    patterns = (
        glob.glob(f"{SOURCE_FOLDER}\\Agent Schedules*.csv")  +
        glob.glob(f"{SOURCE_FOLDER}\\Agent Schedules*.xlsx") +
        glob.glob(f"{SOURCE_FOLDER}\\report*.xlsx")          +
        glob.glob(f"{SOURCE_FOLDER}\\report*.csv")
    )
    moved = False
    for fp in sorted(patterns, key=os.path.getmtime, reverse=True):
        if fp.endswith(".crdownload"): continue
        if os.path.exists(dst_path): os.remove(dst_path)
        shutil.move(fp, dst_path)
        print(f"  📁 {os.path.basename(fp)} → {NICE_FILE}")
        moved = True; break

    if not moved:
        print(f"  ⚠️ No report file found in {SOURCE_FOLDER}")

except RuntimeError as e:
    print(f"\n🚨 {e}")
except WebDriverException as e:
    print(f"\n🚨 WebDriver error: {e}")
finally:
    driver.quit()
    print(f"\n{'═'*55}")
    print(f"✅ NICE IEX Bot finished: {datetime.now().strftime('%d-%b-%Y %H:%M:%S')}")
    print(f"{'═'*55}")


═══════════════════════════════════════════════════════
🚀 NICE IEX Bot started: 25-Aug-2026 23:05:14
📅 Week: 8/24/26 → 8/30/26 | File: 2026_08_24.xlsx
═══════════════════════════════════════════════════════

[1/4] NICE WFM Login...
  🔑 NICE login page detected
  ✅ Entered username: huuchinh.nguyen@concentrix.com
  ✅ Clicked Next (username)
  ✅ Clicked 'Password' method card
  ✅ Entered password
  ✅ Clicked Next (password)
  🎉 NICE login successful

[2/4] Verifying NICE page...
  ✅ NICE page ready (attempt 1)

[3/4] Generating report 8/24/26 → 8/30/26...
  ✅ Generate clicked
  ⏳ Waiting 60s for generation...

[4/4] Downloading Agent Schedules...

🚨 WebDriver error: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6c2a34015+5ba385]
	chromedriver!GetHandleVerifier [0x7ff6c248f0b0+15420]
	chromedriver!(No symbol) [0x7ff6c1fc5b8d]
	chromedriver!(No symbol) [0x7ff6c2020b49]
	chromedriver!(No symbol) [0x7ff6c2020e4c]
	chromedriver!(No symbol) [0x7ff6c2071cc7]
	chromedriver!(No sym